# 01 — Exploración del Corpus Legal

Análisis del corpus `final_json/`: estadísticas, distribución de categorías, longitud de textos y calidad de los datos.

In [ ]:
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Cargar todos los documentos
registros = []
for path in glob.glob('../final_json/**/*.json', recursive=True):
    with open(path, encoding='utf-8') as f:
        entradas = json.load(f)
    for d in entradas:
        d['archivo'] = path
        registros.append(d)

df = pd.DataFrame(registros)
print(f'Total de entradas: {len(df)}')
df.head(3)

In [ ]:
# Distribución por tipo de documento
print('Distribución por tipo_doc:')
print(df['tipo_doc'].value_counts())
print()
print('Archivos por tipo:')
print(df.groupby('tipo_doc')['nombre_doc'].nunique())

In [ ]:
# Distribución de categorías de consumo
todas_cats = []
for cats in df['categoria_consumo']:
    todas_cats.extend(cats)

conteo_cats = Counter(todas_cats)
print('Top categorías de consumo:')
for cat, n in conteo_cats.most_common():
    print(f'  {n:>4}  {cat}')

In [ ]:
# Longitud de textos (en palabras)
df['num_palabras'] = df['texto'].str.split().str.len()

print('Estadísticas de longitud (palabras por entrada):')
print(df['num_palabras'].describe().round(1))

fig, ax = plt.subplots(figsize=(10, 4))
df['num_palabras'].hist(bins=40, ax=ax, edgecolor='white')
ax.set_title('Distribución de longitud de textos')
ax.set_xlabel('Palabras')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Textos más largos (posibles candidatos a chunking)
umbral_tokens = 400  # ~512 tokens aproximado
largos = df[df['num_palabras'] > umbral_tokens]
print(f'Entradas con más de {umbral_tokens} palabras: {len(largos)}')
if len(largos) > 0:
    print('\nDocumentos afectados:')
    print(largos[['nombre_doc', 'num_palabras']].to_string(index=False))

In [ ]:
# Muestra de texto: comparar fuente legal vs. lo que verá el LLM
muestra = df[df['tipo_doc'] == 'ley'].iloc[0]
print(f'Documento: {muestra["nombre_doc"]}')
print(f'Sección:   {muestra["capitulo_seccion"]}')
print(f'Palabras:  {muestra["num_palabras"]}')
print()
print('--- TEXTO LEGAL ---')
print(muestra['texto'])